# ALeRCE Period Calculator

This notebook allows you to calculate periods for astronomical objects from the ALeRCE database. You can:

1. Search for objects by OID or filter by survey_class_mapped
2. Calculate periods using the PeriodExtractor class
3. Visualize light curves (folded and unfolded)
4. Work with data from partitions.parquet files

## 1. Setup and Environment Configuration

First, let's import all the necessary libraries and set up our environment.

In [13]:
# Import necessary libraries
import sys
sys.path.append('../')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Using regular tqdm instead of tqdm.notebook to avoid ipywidgets dependency
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Special imports for ALeRCE data processing
from lc_classifier.features.core.base import astro_object_from_dict, AstroObject
from lc_classifier.features.extractors.period_extractor import PeriodExtractor
from lc_classifier.features.preprocess.ztf import ZTFLightcurvePreprocessor

# Set plot style
plt.style.use('ggplot')

## 2. Load Data

Let's load the objects file that contains the survey_class_mapped field. We'll check its structure to understand what information is available.

In [14]:
# Path to objects file (already referenced in your existing notebook)
objects_file = '/home/fsoto/Documents/LCsSSL/data/alerce_data_raw/ts_v9.0.1_b3000_objs_250408_ndetge8.parquet'
data_folder = "/home/fsoto/Documents/LCsSSL/data/alerce_data_raw/data_250408_ndetge8_ao"

# Load objects file if not already loaded
if 'objects' not in locals():
    print("Loading objects file...")
    objects = pd.read_parquet(objects_file)
    print(f"Loaded {len(objects)} objects")
else:
    print(f"Objects file already loaded with {len(objects)} entries")

# Display columns and first few rows
print("\nColumns in objects file:")
print(objects.columns.tolist())

print("\nFirst 5 rows:")
display(objects.head())

# Find unique classes in survey_class_mapped if it exists
if 'survey_class_mapped' in objects.columns:
    print("\nUnique classes in survey_class_mapped:")
    display(objects['survey_class_mapped'].value_counts())
elif 'alerceclass' in objects.columns:  # Alternative column name 
    print("\nUnique classes in alerceclass:")
    display(objects['alerceclass'].value_counts())
else:
    # Try to find the column that contains class information
    class_columns = [col for col in objects.columns if 'class' in col.lower()]
    if class_columns:
        print(f"\nPossible class columns found: {class_columns}")
        for col in class_columns:
            print(f"\nUnique values in {col}:")
            display(objects[col].value_counts())
    else:
        print("\nNo class column found in the objects file")

Objects file already loaded with 39483 entries

Columns in objects file:
['oid', 'meanra', 'meandec', 'survey_class_mapped', 'firstmjd', 'mjdmax']

First 5 rows:


,oid,meanra,meandec,survey_class_mapped,firstmjd,mjdmax
0,ZTF17aaazlzl,120.870621,33.462329,AGN,58363.482743,60254.0
1,ZTF21acqoruu,297.425121,3.303783,AGN,59623.559595,60254.0
2,ZTF22aaafnol,165.188738,62.921204,AGN,59671.362998,60254.0
3,ZTF22aaaokdq,219.567335,-6.972391,AGN,59623.400185,60254.0
4,ZTF22aacbgva,79.088411,-10.561478,AGN,59622.148692,60254.0



Unique classes in survey_class_mapped:


survey_class_mapped
RRLc              2936
RRLab             2931
LPV               2931
QSO               2923
EB/EW             2921
EA                2909
AGN               2898
CEP               2891
YSO               2773
SNIa              2486
RSCVn             2452
CV/Nova           1640
DSCT              1610
Blazar            1393
SNII              1393
Periodic-Other    1269
SNIbc              452
SNIIn              267
SLSN               185
SNIIb              113
TDE                 81
Microlensing        29
Name: count, dtype: int64

## 3. Find AstroObject Batch Files

Now let's locate and identify the available AstroObject batch files.

In [15]:
# Find all available astro object batch files
try:
    astro_objects_filenames = os.listdir(data_folder)
    astro_objects_filenames = [
        f for f in astro_objects_filenames if "astro_objects_batch" in f
    ]
    print(f"Found {len(astro_objects_filenames)} batch files")
    if len(astro_objects_filenames) > 0:
        print(f"Example files: {astro_objects_filenames[:3]}")
except FileNotFoundError:
    print(f"Data folder '{data_folder}' not found. Please check the path.")
    astro_objects_filenames = []

Found 105 batch files
Example files: ['astro_objects_batch_006.pkl', 'astro_objects_batch_072.pkl', 'astro_objects_batch_084.pkl']


## 4. Configure the PeriodExtractor

Let's define a function to create the PeriodExtractor with customizable parameters.

In [16]:
def create_period_extractor(
    bands=['g', 'r'], 
    unit="magnitude",
    smallest_period=0.045,  # ~1.2 hours
    largest_period=100.0,  # 1000 days
    trim_lightcurve_to_n_days=None,
    min_length=15,
    use_forced_photo=True,
    return_power_rates=True,
    shift=0.1
):
    """
    Create a PeriodExtractor instance with specified parameters.
    
    Parameters:
    -----------
    bands : list
        List of bands to use for period calculation
    unit : str
        Unit of brightness measurement ('magnitude' or 'diff_flux')
    smallest_period : float
        Smallest period to search for in days
    largest_period : float
        Largest period to search for in days
    trim_lightcurve_to_n_days : float or None
        Trim light curve to this many days or None for no trimming
    min_length : int
        Minimum number of observations required per band
    use_forced_photo : bool
        Whether to use forced photometry if available
    return_power_rates : bool
        Whether to calculate and return power rates
    shift : float
        Shift parameter for frequency grid evaluation
    
    Returns:
    --------
    PeriodExtractor instance
    """
    return PeriodExtractor(
        bands=bands,
        unit=unit,
        smallest_period=smallest_period,
        largest_period=largest_period,
        trim_lightcurve_to_n_days=trim_lightcurve_to_n_days,
        min_length=min_length,
        use_forced_photo=use_forced_photo,
        return_power_rates=return_power_rates,
        shift=shift
    )

## 5. Load and Convert AstroObjects

Define functions to load and convert batch files to AstroObjects.

In [17]:
def load_astro_objects_batch(batch_filename):
    """
    Load a batch of AstroObjects from a pickle file.
    
    Parameters:
    -----------
    batch_filename : str
        Name of the batch file to load
        
    Returns:
    --------
    List of AstroObject instances
    """
    batch_path = os.path.join(data_folder, batch_filename)
    if not os.path.exists(batch_path):
        print(f"Batch file not found: {batch_path}")
        return None
        
    batch_astro_objects = pd.read_pickle(batch_path)
    batch_astro_objects = [astro_object_from_dict(d) for d in batch_astro_objects]
    
    # Preprocess light curves
    lightcurve_preprocessor = ZTFLightcurvePreprocessor()
    lightcurve_preprocessor.preprocess_batch(batch_astro_objects)
    
    print(f"Loaded {len(batch_astro_objects)} objects from {batch_filename}")
    return batch_astro_objects

## 6. Create a Mapping between OIDs and Batch Files

To efficiently find objects by OID, let's create a mapping between OIDs and their corresponding batch files.

In [18]:
def create_oid_to_batch_mapping(max_batches=None):
    """
    Create a mapping between OIDs and their batch files.
    
    Parameters:
    -----------
    max_batches : int or None
        Maximum number of batch files to process (None for all)
        
    Returns:
    --------
    dict
        Mapping from OID to batch filename
    """
    oid_to_batch = {}
    processed_batches = 0
    
    # Process batch files
    batch_files = astro_objects_filenames[:max_batches] if max_batches else astro_objects_filenames
    
    print(f"Processing {len(batch_files)} batch files")
    for i, batch_file in enumerate(batch_files):
        if i % 10 == 0:  # Print status every 10 files
            print(f"Processing batch {i+1}/{len(batch_files)}: {batch_file}")
        
        try:
            # Load just one batch at a time to save memory
            batch_path = os.path.join(data_folder, batch_file)
            batch_astro_objects = pd.read_pickle(batch_path)
            
            # Extract OIDs from batch
            for i, obj_dict in enumerate(batch_astro_objects):
                # Convert to AstroObject to access metadata
                obj = astro_object_from_dict(obj_dict)
                
                # Get OID from metadata
                if "oid" in obj.metadata["name"].values:
                    oid_row = obj.metadata[obj.metadata["name"] == "oid"]
                    oid = oid_row["value"].values[0]
                    oid_to_batch[oid] = batch_file
                elif "aid" in obj.metadata["name"].values:
                    aid_row = obj.metadata[obj.metadata["name"] == "aid"]
                    oid = aid_row["value"].values[0]
                    oid_to_batch[oid] = batch_file
            
            processed_batches += 1
            
        except Exception as e:
            print(f"Error processing batch {batch_file}: {str(e)}")
    
    print(f"Processed {processed_batches} batches")
    print(f"Found {len(oid_to_batch)} unique OIDs")
    
    return oid_to_batch

# Create the mapping (optional - can take time for large datasets)
# Uncomment the next line to create the mapping when needed
oid_to_batch_mapping = create_oid_to_batch_mapping(max_batches=5)  # Limit to 5 batches for testing

Processing 5 batch files
Processing batch 1/5: astro_objects_batch_006.pkl


Processed 5 batches
Found 1849 unique OIDs


## 6.1 Create Enhanced OID to Batch and Position Mapping

Let's create a more efficient mapping that stores not only the batch file but also the object's position within the batch.

In [19]:
def create_enhanced_oid_mapping(max_batches=None, save_mapping=True, load_if_exists=True, mapping_path='oid_mapping.pkl'):
    """
    Create an enhanced mapping between OIDs and their batch files + positions.
    
    Parameters:
    -----------
    max_batches : int or None
        Maximum number of batch files to process (None for all)
    save_mapping : bool
        Whether to save the mapping to disk
    load_if_exists : bool
        Whether to load existing mapping from disk if available
    mapping_path : str
        Path to save/load the mapping
        
    Returns:
    --------
    dict
        Mapping from OID to tuple of (batch_filename, position_in_batch)
    """
    # Import pickle at the start
    import pickle
    
    # Check if mapping exists and load_if_exists is True
    if load_if_exists and os.path.exists(mapping_path):
        print(f"Loading existing mapping from {mapping_path}")
        try:
            with open(mapping_path, 'rb') as f:
                mapping = pickle.load(f)
            print(f"Loaded mapping with {len(mapping)} entries")
            return mapping
        except Exception as e:
            print(f"Error loading mapping: {str(e)}")
    
    # Create new mapping
    oid_mapping = {}
    processed_batches = 0
    
    # Process batch files
    batch_files = astro_objects_filenames[:max_batches] if max_batches else astro_objects_filenames
    
    print(f"Processing {len(batch_files)} batch files...")
    for batch_idx, batch_file in enumerate(batch_files):
        if batch_idx % 10 == 0:  # Print status periodically
            print(f"Processing batch {batch_idx+1}/{len(batch_files)}: {batch_file}")
        
        try:
            # Load batch
            batch_path = os.path.join(data_folder, batch_file)
            batch_astro_objects = pd.read_pickle(batch_path)
            
            # Process each object
            for obj_idx, obj_dict in enumerate(batch_astro_objects):
                # Convert to AstroObject to access metadata
                obj = astro_object_from_dict(obj_dict)
                
                # Extract OID
                for id_field in ["oid", "aid"]:
                    if id_field in obj.metadata["name"].values:
                        id_row = obj.metadata[obj.metadata["name"] == id_field]
                        oid = id_row["value"].values[0]
                        oid_mapping[oid] = (batch_file, obj_idx)
                        break
            
            processed_batches += 1
            
        except Exception as e:
            print(f"Error processing batch {batch_file}: {str(e)}")
    
    print(f"Processed {processed_batches} batches")
    print(f"Created mapping with {len(oid_mapping)} unique OIDs")
    
    # Save mapping if requested
    if save_mapping:
        print(f"Saving mapping to {mapping_path}")
        try:
            with open(mapping_path, 'wb') as f:
                pickle.dump(oid_mapping, f)
            print("Mapping saved successfully")
        except Exception as e:
            print(f"Error saving mapping: {str(e)}")
    
    return oid_mapping

def get_object_by_oid(oid, oid_mapping, add_class_info=True):
    """
    Get an object directly by its OID using the enhanced mapping.
    
    Parameters:
    -----------
    oid : str
        Object ID to retrieve
    oid_mapping : dict
        Mapping from OID to (batch_filename, position_in_batch)
    add_class_info : bool
        Whether to add classification info from the objects dataframe if available
        
    Returns:
    --------
    Tuple of (AstroObject, class_info_dict) or (None, None)
        Retrieved object and classification info if found, None otherwise
    """
    if oid not in oid_mapping:
        print(f"OID {oid} not found in mapping")
        return None, None
    
    batch_file, obj_idx = oid_mapping[oid]
    print(f"Found OID {oid} in batch {batch_file} at position {obj_idx}")
    
    # Try to get class information if add_class_info is True
    class_info = {}
    if add_class_info and 'objects' in globals():
        # Check if the objects dataframe contains this OID
        oid_row = None
        for id_field in ['oid', 'aid', 'object_id']:
            if id_field in objects.columns:
                oid_row = objects[objects[id_field] == oid]
                if not oid_row.empty:
                    break
        
        # Extract class information if found
        if oid_row is not None and not oid_row.empty:
            for class_field in ['survey_class_mapped', 'alerceclass', 'class_name', 'class']:
                if class_field in oid_row.columns:
                    class_info['class'] = oid_row[class_field].iloc[0]
                    break
            
            # Add additional metadata if available
            for meta_field in ['meanra', 'meandec', 'firstmjd', 'lastmjd', 'ndet']:
                if meta_field in oid_row.columns:
                    class_info[meta_field] = oid_row[meta_field].iloc[0]
    
    # Load batch
    batch_path = os.path.join(data_folder, batch_file)
    try:
        batch_astro_objects = pd.read_pickle(batch_path)
        
        # Get specific object
        if obj_idx < len(batch_astro_objects):
            obj_dict = batch_astro_objects[obj_idx]
            obj = astro_object_from_dict(obj_dict)
            
            # Preprocess
            lightcurve_preprocessor = ZTFLightcurvePreprocessor()
            lightcurve_preprocessor.preprocess_batch([obj])  # Use preprocess_batch instead of preprocess
            
            return obj, class_info
        else:
            print(f"Object index {obj_idx} is out of range for batch {batch_file}")
            return None, None
    except Exception as e:
        print(f"Error retrieving object: {str(e)}")
        return None, None

## 7. Visualize Light Curves and Periods

Define functions to visualize light curves and phase-folded light curves.

In [20]:
def plot_light_curve(astro_object, period=None, title=None):
    """
    Plot the light curve of an astro object and optionally fold it by period.
    
    Parameters:
    -----------
    astro_object : AstroObject
        The astronomical object to plot
    period : float or None
        Period to fold the light curve by (if None, no folding)
    title : str or None
        Title for the plot
    """
    # Get observations
    observations = astro_object.detections
    
    if observations is None or len(observations) == 0:
        print("No observations available for this object")
        return
    
    # Filter for magnitude observations
    obs = observations[observations['unit'] == 'magnitude']
    
    if len(obs) == 0:
        print("No magnitude observations available for this object")
        return
    
    # Create figure
    if period is not None and not np.isnan(period):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    else:
        fig, ax1 = plt.subplots(figsize=(10, 5))
        ax2 = None
    
    # Get unique bands
    bands = obs['fid'].unique()
    colors = {'g': 'green', 'r': 'red', 'i': 'purple'}
    
    # Plot each band
    for band in bands:
        band_data = obs[obs['fid'] == band]
        color = colors.get(band, 'blue')
        
        # Regular light curve
        ax1.errorbar(
            band_data['mjd'], 
            band_data['brightness'],
            yerr=band_data['e_brightness'], 
            fmt='o', 
            color=color,
            label=f'Band {band}',
            alpha=0.7
        )
        
        # Phase-folded light curve if period is provided
        if ax2 is not None and period is not None and not np.isnan(period):
            phases = (band_data['mjd'] % period) / period
            ax2.errorbar(
                phases, 
                band_data['brightness'],
                yerr=band_data['e_brightness'], 
                fmt='o', 
                color=color,
                label=f'Band {band}',
                alpha=0.7
            )
            # Plot second cycle for clarity
            ax2.errorbar(
                phases + 1, 
                band_data['brightness'],
                yerr=band_data['e_brightness'], 
                fmt='o', 
                color=color,
                alpha=0.4
            )
    
    # Configure axes
    ax1.set_xlabel('MJD')
    ax1.set_ylabel('Magnitude')
    ax1.set_title('Light Curve')
    ax1.invert_yaxis()  # Magnitudes decrease upward
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    if title:
        fig.suptitle(title, fontsize=14)
    
    # Configure phase-folded plot if available
    if ax2 is not None:
        ax2.set_xlabel('Phase')
        ax2.set_ylabel('Magnitude')
        ax2.set_title(f'Phase-folded (Period = {period:.4f} days)')
        ax2.set_xlim(0, 2)
        ax2.invert_yaxis()  # Magnitudes decrease upward
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Calculate Period for a Specific Object

Define a function to extract the period for a specific AstroObject.

In [21]:
def extract_period_for_object(astro_object, plot=True, max_period=100.0):
    """
    Extract period for a specific AstroObject and optionally plot the results.
    
    Parameters:
    -----------
    astro_object : AstroObject
        The AstroObject to process
    plot : bool
        Whether to plot the results
    max_period : float
        Maximum period to search for in days (default: 100.0)
    
    Returns:
    --------
    Tuple of (period, significance)
    """
    # Create period extractor with specified maximum period
    period_extractor = create_period_extractor(largest_period=max_period)
    
    # Clean existing period features if any
    if astro_object.features is not None and not astro_object.features.empty:
        period_features = [
            'Multiband_period', 'PPE', 'Period_band', 'delta_period'
        ] + period_extractor.pr_names
        astro_object.features = astro_object.features[
            ~astro_object.features['name'].isin(period_features)
        ]
    
    # Extract period features
    period_extractor.compute_features_single_object(astro_object)
    
    # Get period and significance
    period_features = astro_object.features[astro_object.features['name'] == 'Multiband_period']
    ppe_features = astro_object.features[astro_object.features['name'] == 'PPE']
    
    period = period_features['value'].values[0] if len(period_features) > 0 else np.nan
    significance = ppe_features['value'].values[0] if len(ppe_features) > 0 else np.nan
    
    # Plot if requested
    if plot and not np.isnan(period):
        plot_light_curve(astro_object, period, title=f"Object {get_object_id(astro_object)}")
    
    return period, significance

def get_object_id(astro_object):
    """Extract the object ID from an AstroObject's metadata"""
    for id_field in ["oid", "aid"]:
        if id_field in astro_object.metadata["name"].values:
            id_row = astro_object.metadata[astro_object.metadata["name"] == id_field]
            return id_row["value"].values[0]
    return "unknown"

## 9. Search for Objects by Class and Calculate Periods

Now let's create functions to search for objects by class and OID, and calculate their periods.

In [22]:
def search_objects_by_class(class_name, sample_size=5, random_seed=42):
    """
    Search for objects by class in the objects dataframe.
    
    Parameters:
    -----------
    class_name : str
        Class name to search for
    sample_size : int
        Number of objects to sample
    random_seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    DataFrame
        Sampled objects from the specified class
    """
    # Identify the class column in the objects dataframe
    class_column = None
    for col in ['survey_class_mapped', 'alerceclass', 'class_name', 'class']:
        if col in objects.columns:
            class_column = col
            break
    
    if class_column is None:
        print("No class column found in the objects dataframe")
        return None
    
    # Filter objects by class
    class_objects = objects[objects[class_column] == class_name]
    
    if len(class_objects) == 0:
        print(f"No objects found for class {class_name}")
        return None
        
    print(f"Found {len(class_objects)} objects with class {class_name}")
    
    # Sample objects
    if sample_size and sample_size < len(class_objects):
        np.random.seed(random_seed)
        sampled_objects = class_objects.sample(sample_size)
        print(f"Sampled {sample_size} objects")
    else:
        sampled_objects = class_objects
        print(f"Using all {len(class_objects)} objects")
        
    return sampled_objects

def find_and_calculate_period(oid, oid_to_batch_mapping=None, max_period=100.0):
    """
    Find an object by OID and calculate its period.
    
    Parameters:
    -----------
    oid : str
        Object ID to search for
    oid_to_batch_mapping : dict or None
        Mapping from OID to batch filename (if None, will search all batches)
    max_period : float
        Maximum period to search for in days (default: 100.0)
    
    Returns:
    --------
    Tuple of (period, significance)
    """
    batch_file = None
    
    # Use mapping if available
    if oid_to_batch_mapping and oid in oid_to_batch_mapping:
        batch_file = oid_to_batch_mapping[oid]
        print(f"Found OID {oid} in batch file {batch_file}")
        
        # Load batch and find object
        batch_objects = load_astro_objects_batch(batch_file)
        if batch_objects:
            for obj in batch_objects:
                obj_id = get_object_id(obj)
                if obj_id == oid:
                    print(f"Found object {oid} in loaded batch")
                    period, significance = extract_period_for_object(obj, max_period=max_period)
                    print(f"Period: {period:.4f} days, Significance: {significance:.4f}")
                    return period, significance
    
    # If not found in mapping or mapping not provided, search all batches
    print(f"Searching for OID {oid} in all batch files...")
    
    total_batches = len(astro_objects_filenames)
    for i, batch_file in enumerate(astro_objects_filenames):
        if i % 10 == 0:  # Print status every 10 files
            print(f"Checking batch {i+1}/{total_batches}")
        
        batch_objects = load_astro_objects_batch(batch_file)
        if not batch_objects:
            continue
            
        for obj in batch_objects:
            obj_id = get_object_id(obj)
            if obj_id == oid:
                print(f"Found object {oid} in batch {batch_file}")
                period, significance = extract_period_for_object(obj)
                print(f"Period: {period:.4f} days, Significance: {significance:.4f}")
                return period, significance
    
    print(f"Object {oid} not found in any batch file")
    return None, None

## 10. Interactive Period Calculation Interface

Let's create functions for an interactive interface to calculate periods for objects from specific classes or by OID.

In [23]:
def generate_class_statistics():
    """
    Generate statistics about the available classes in the objects dataframe.
    
    Returns:
    --------
    pandas.DataFrame
        Class statistics
    """
    # Identify the class column in the objects dataframe
    class_column = None
    for col in ['survey_class_mapped', 'alerceclass', 'class_name', 'class']:
        if col in objects.columns:
            class_column = col
            break
    
    if class_column is None:
        print("No class column found in the objects dataframe")
        return None
        
    # Count objects by class
    class_counts = objects[class_column].value_counts()
    
    # Create statistics dataframe
    stats = pd.DataFrame({
        'Class': class_counts.index,
        'Count': class_counts.values,
        'Percentage': (class_counts.values / len(objects) * 100).round(2)
    })
    
    return stats

def process_random_objects_by_class(class_name, num_objects=3, oid_to_batch_mapping=None, max_period=100.0):
    """
    Process random objects from a specific class.
    
    Parameters:
    -----------
    class_name : str
        Class name to search for
    num_objects : int
        Number of random objects to process
    oid_to_batch_mapping : dict or None
        Mapping from OID to batch filename
    max_period : float
        Maximum period to search for in days (default: 100.0)
    """
    # Search for objects by class
    sampled_objects = search_objects_by_class(class_name, sample_size=num_objects)
    
    if sampled_objects is None or len(sampled_objects) == 0:
        return
    
    # Process each sampled object
    results = []
    for i, (_, obj_row) in enumerate(sampled_objects.iterrows()):
        # Get OID field
        oid = None
        for oid_field in ['oid', 'aid', 'object_id']:
            if oid_field in obj_row:
                oid = obj_row[oid_field]
                break
        
        if oid is None:
            print(f"No OID found for object at index {i}")
            continue
            
        print(f"\nProcessing object {i+1}/{len(sampled_objects)}: {oid}")
        period, significance = find_and_calculate_period(oid, oid_to_batch_mapping, max_period=max_period)
        
        results.append({
            'OID': oid,
            'Period': period,
            'Significance': significance
        })
    
    # Display results
    return pd.DataFrame(results)

## 11. Main Interactive Interface

Now let's use our functions to create an interactive interface for exploring objects and calculating periods.

In [24]:
# Display class statistics
class_stats = generate_class_statistics()
if class_stats is not None:
    print("Class Distribution in the Dataset:")
    display(class_stats)

# If you want to create a mapping between OIDs and batch files (takes time)
# Uncomment the next line to build the mapping
oid_to_batch_mapping = create_oid_to_batch_mapping(max_batches=None)

# Function to select a random object from a given class and calculate its period
def calculate_period_for_class(class_name, num_objects=1, random_seed=None, oid_to_batch_mapping=None, max_period=100.0):
    """
    Calculate periods for random objects from a specific class.
    
    Parameters:
    -----------
    class_name : str
        Class name to search for
    num_objects : int
        Number of random objects to process
    random_seed : int or None
        Random seed for reproducibility
    oid_to_batch_mapping : dict or None
        Mapping from OID to batch file
    max_period : float
        Maximum period to search for in days (default: 100.0)
    """
    results = process_random_objects_by_class(class_name, num_objects=num_objects, oid_to_batch_mapping=oid_to_batch_mapping, max_period=max_period)
    if results is not None:
        display(results)
        
# Function to get period for a specific OID
def calculate_period_for_oid(oid, oid_to_batch_mapping=None, max_period=100.0):
    """
    Calculate period for a specific OID.
    
    Parameters:
    -----------
    oid : str
        Object ID to search for
    oid_to_batch_mapping : dict
        Mapping from OID to batch filename
    max_period : float
        Maximum period to search for in days (default: 100.0)
    """
    period, significance = find_and_calculate_period(oid, oid_to_batch_mapping=oid_to_batch_mapping, max_period=max_period)
    if period is not None:
        return period, significance
    else:
        return None, None

Class Distribution in the Dataset:


,Class,Count,Percentage
0,RRLc,2936,7.44
1,RRLab,2931,7.42
2,LPV,2931,7.42
3,QSO,2923,7.40
4,EB/EW,2921,7.40
5,EA,2909,7.37
6,AGN,2898,7.34
7,CEP,2891,7.32
8,YSO,2773,7.02
9,SNIa,2486,6.30


Processing 105 batch files
Processing batch 1/105: astro_objects_batch_006.pkl
Processing batch 11/105: astro_objects_batch_002.pkl
Processing batch 21/105: astro_objects_batch_064.pkl
Processing batch 31/105: astro_objects_batch_059.pkl
Processing batch 41/105: astro_objects_batch_065.pkl
Processing batch 51/105: astro_objects_batch_086.pkl
Processing batch 61/105: astro_objects_batch_016.pkl
Processing batch 71/105: astro_objects_batch_040.pkl
Processing batch 81/105: astro_objects_batch_071.pkl
Processing batch 91/105: astro_objects_batch_041.pkl
Processing batch 101/105: astro_objects_batch_030.pkl
Processed 105 batches
Found 39483 unique OIDs


## 12. Examples - Try These!

Here are some examples of using the interactive interface to calculate periods for objects.

In [25]:
def get_oids_by_class(class_name, limit=None, random_sample=False, random_seed=42):
    """
    Retrieve OIDs for objects belonging to a specific class.
    
    Parameters:
    -----------
    class_name : str
        The class to filter by (must exist in survey_class_mapped)
    limit : int or None
        Maximum number of OIDs to return (None returns all)
    random_sample : bool
        If True and limit is provided, returns a random sample instead of the first N objects
    random_seed : int
        Random seed for reproducibility when random_sample is True
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame containing OIDs and basic information for the requested class
    """
    # Check if the class exists
    if class_name not in objects['survey_class_mapped'].unique():
        print(f"Class '{class_name}' not found. Available classes:")
        print(objects['survey_class_mapped'].unique())
        return None
    
    # Filter objects by class
    filtered_objects = objects[objects['survey_class_mapped'] == class_name]
    print(f"Found {len(filtered_objects)} objects with class '{class_name}'")
    
    # Apply limit
    if limit is not None and limit < len(filtered_objects):
        if random_sample:
            # Random sample
            np.random.seed(random_seed)
            filtered_objects = filtered_objects.sample(limit)
            print(f"Randomly sampled {limit} objects")
        else:
            # First N objects
            filtered_objects = filtered_objects.head(limit)
            print(f"Returning first {limit} objects")
    
    # Return results
    return filtered_objects[['oid', 'meanra', 'meandec', 'firstmjd']]

# Example usage:
# Get 5 random RRL objects
rrl_oids = get_oids_by_class('LPV', limit=5, random_sample=True)
if rrl_oids is not None:
    display(rrl_oids)

Found 2931 objects with class 'LPV'
Randomly sampled 5 objects


,oid,meanra,meandec,firstmjd
17619,ZTF18adoedvk,305.361403,9.572866,59673.472303
18629,ZTF17aaaajra,317.160055,44.255624,58274.471042
19084,ZTF18abaqgow,2.525315,58.222101,58283.386088
18388,ZTF18aayfabj,332.522140,63.338290,58274.469641
17806,ZTF18abyisyl,351.323015,60.958911,58357.219306


In [26]:
# Example 1: Calculate periods for 3 random objects from a specific class
# Note: Update 'LPV' to an actual class name from your dataset

# Default maximum period (100.0 days)
#calculate_period_for_class('LPV', num_objects=1, oid_to_batch_mapping=oid_to_batch_mapping)

# With custom maximum period - useful for different types of variable stars
# Uncomment the line below to try a different maximum period:
# calculate_period_for_class('LPV', num_objects=1, oid_to_batch_mapping=oid_to_batch_mapping, max_period=200.0)

# Example 2: Calculate period for a specific OID
# Note: Replace 'ZTF19abmolyr' with an actual OID from your dataset
#calculate_period_for_oid('ZTF17aaaajra', oid_to_batch_mapping=oid_to_batch_mapping)

# Example 3: Look at the first few OIDs in the objects dataframe
# print("\nSample OIDs from the objects dataframe:")
# if 'oid' in objects.columns:
#     print(objects['oid'].iloc[:5].tolist())
# elif 'aid' in objects.columns:
#     print(objects['aid'].iloc[:5].tolist())
# elif 'object_id' in objects.columns:
#     print(objects['object_id'].iloc[:5].tolist())

# To use these examples, uncomment the appropriate lines and run the cell

## 13. Preload All AstroObjects for Faster Processing

To speed up period calculations when working with multiple objects, we'll load all AstroObjects into memory beforehand.

In [27]:
def load_all_astro_objects(max_batches=None, show_progress=True, save_to_cache=True, cache_path='all_astro_objects_cache.pkl'):
    """
    Load all AstroObjects from batch files into memory for faster access.
    
    Parameters:
    -----------
    max_batches : int or None
        Maximum number of batch files to load (None for all)
    show_progress : bool
        Whether to show a progress bar
    save_to_cache : bool
        Whether to save loaded objects to a cache file
    cache_path : str
        Path to cache file
        
    Returns:
    --------
    dict
        Dictionary mapping OIDs to their AstroObjects
    """
    import pickle
    import time
    
    # Check if cache exists and try to load it
    if os.path.exists(cache_path):
        print(f"Found cache file at {cache_path}. Attempting to load...")
        try:
            with open(cache_path, 'rb') as f:
                all_objects = pickle.load(f)
                print(f"Successfully loaded {len(all_objects)} objects from cache.")
                return all_objects
        except Exception as e:
            print(f"Error loading from cache: {str(e)}")
            print("Proceeding with loading from batch files.")
    
    all_objects = {}
    start_time = time.time()
    
    # Get batch files to process
    batch_files = astro_objects_filenames[:max_batches] if max_batches else astro_objects_filenames
    total_batches = len(batch_files)
    
    print(f"Loading objects from {total_batches} batch files...")
    
    # Set up progress tracking
    if show_progress:
        batch_iterator = tqdm(batch_files)
    else:
        batch_iterator = batch_files
        
    # Process each batch
    lightcurve_preprocessor = ZTFLightcurvePreprocessor()
    for batch_file in batch_iterator:
        try:
            # Load batch
            batch_path = os.path.join(data_folder, batch_file)
            batch_astro_objects = pd.read_pickle(batch_path)
            
            # Convert and preprocess
            batch_objects = [astro_object_from_dict(d) for d in batch_astro_objects]
            lightcurve_preprocessor.preprocess_batch(batch_objects)
            
            # Store in dictionary by OID
            for obj in batch_objects:
                oid = get_object_id(obj)
                if oid != "unknown":
                    all_objects[oid] = obj
                    
        except Exception as e:
            print(f"Error loading batch {batch_file}: {str(e)}")
    
    elapsed_time = time.time() - start_time
    print(f"Loaded {len(all_objects)} objects in {elapsed_time:.2f} seconds.")
    
    # Save to cache if requested
    if save_to_cache:
        print(f"Saving objects to cache at {cache_path}...")
        try:
            with open(cache_path, 'wb') as f:
                pickle.dump(all_objects, f)
            print("Successfully saved to cache.")
        except Exception as e:
            print(f"Error saving to cache: {str(e)}")
    
    return all_objects

def get_object_by_oid_from_preloaded(oid, all_objects, add_class_info=True):
    """
    Get an object by OID from the preloaded objects dictionary.
    
    Parameters:
    -----------
    oid : str
        Object ID to retrieve
    all_objects : dict
        Dictionary mapping OIDs to AstroObjects
    add_class_info : bool
        Whether to add class information from objects dataframe
        
    Returns:
    --------
    Tuple of (AstroObject, class_info_dict) or (None, None)
    """
    if oid not in all_objects:
        print(f"OID {oid} not found in preloaded objects")
        return None, None
    
    obj = all_objects[oid]
    print(f"Found OID {oid} in preloaded objects")
    
    # Add class information if requested
    class_info = {}
    if add_class_info and 'objects' in globals():
        # Check if objects dataframe contains this OID
        oid_row = None
        for id_field in ['oid', 'aid', 'object_id']:
            if id_field in objects.columns:
                oid_row = objects[objects[id_field] == oid]
                if not oid_row.empty:
                    break
        
        # Extract class information if found
        if oid_row is not None and not oid_row.empty:
            for class_field in ['survey_class_mapped', 'alerceclass', 'class_name', 'class']:
                if class_field in oid_row.columns:
                    class_info['class'] = oid_row[class_field].iloc[0]
                    break
            
            # Add additional metadata
            for meta_field in ['meanra', 'meandec', 'firstmjd', 'lastmjd', 'ndet']:
                if meta_field in oid_row.columns:
                    class_info[meta_field] = oid_row[meta_field].iloc[0]
    
    return obj, class_info

def calculate_period_with_preloaded_objects(oid, all_objects, max_period=100.0):
    """
    Calculate period for an object using preloaded AstroObjects.
    
    Parameters:
    -----------
    oid : str
        Object ID to calculate period for
    all_objects : dict
        Dictionary mapping OIDs to AstroObjects
    max_period : float
        Maximum period to search for in days (default: 100.0)
        
    Returns:
    --------
    Tuple of (period, significance)
    """
    obj, class_info = get_object_by_oid_from_preloaded(oid, all_objects)
    if obj is None:
        print(f"Object {oid} not found in preloaded objects")
        return None, None
    
    # Calculate period with specified maximum period
    period, significance = extract_period_for_object(obj, max_period=max_period)
    
    # Display class information if available
    if class_info and 'class' in class_info:
        print(f"Class: {class_info['class']}")
    
    print(f"Period: {period:.4f} days, Significance: {significance:.4f}")
    return period, significance

def process_objects_by_class_with_preloaded(class_name, num_objects=3, all_objects=None, max_period=100.0):
    """
    Process objects from a specific class using preloaded AstroObjects.
    
    Parameters:
    -----------
    class_name : str
        Class name to search for
    num_objects : int
        Number of objects to process
    all_objects : dict
        Dictionary mapping OIDs to AstroObjects
    max_period : float
        Maximum period to search for in days (default: 100.0)
        
    Returns:
    --------
    pandas.DataFrame
        Results with periods and significances
    """
    if all_objects is None:
        print("No preloaded objects provided. Please load objects first.")
        return None
    
    # Search for objects by class
    sampled_objects = search_objects_by_class(class_name, sample_size=num_objects)
    
    if sampled_objects is None or len(sampled_objects) == 0:
        return None
    
    # Process each sampled object
    results = []
    for i, (_, obj_row) in enumerate(sampled_objects.iterrows()):
        # Get OID field
        oid = None
        for oid_field in ['oid', 'aid', 'object_id']:
            if oid_field in obj_row:
                oid = obj_row[oid_field]
                break
        
        if oid is None:
            print(f"No OID found for object at index {i}")
            continue
            
        print(f"\nProcessing object {i+1}/{len(sampled_objects)}: {oid}")
        period, significance = calculate_period_with_preloaded_objects(oid, all_objects, max_period=max_period)
        
        results.append({
            'OID': oid,
            'Period': period,
            'Significance': significance
        })
    
    # Display results
    return pd.DataFrame(results)

In [48]:
# Load all AstroObjects into memory
# Note: This can take a long time depending on the size of your dataset.
# Uncomment the line below to load all objects, or adjust max_batches for testing
# all_astro_objects = load_all_astro_objects(max_batches=None)

# For demonstration purposes, let's load a smaller subset
all_astro_objects = load_all_astro_objects(max_batches=1)

# Check how many objects were loaded
print(f"Loaded {len(all_astro_objects)} AstroObjects into memory")

Loading objects from 1 batch files...


100%|██████████| 1/1 [01:01<00:00, 61.65s/it]


Loaded 385 objects in 61.65 seconds.
Saving objects to cache at all_astro_objects_cache.pkl...
Successfully saved to cache.
Loaded 385 AstroObjects into memory


## 14. Period Calculation with Preloaded Objects

Using preloaded objects significantly speeds up period calculations for multiple objects.

In [ ]:
def compare_period_calculation_performance(oid, max_period=1000.0):
    """
    Compare the performance of period calculation with and without preloaded objects.
    
    Parameters:
    -----------
    oid : str
        Object ID to use for comparison
    max_period : float
        Maximum period to search for in days (default: 100.0)
    """
    import time
    
    print(f"Comparing period calculation performance for OID {oid} (max_period={max_period})")
    
    # Method 1: Without preloaded objects
    start_time = time.time()
    period1, significance1 = find_and_calculate_period(oid, oid_to_batch_mapping, max_period=max_period)
    elapsed_time1 = time.time() - start_time
    print(f"\nMethod 1 (Without preloading): {elapsed_time1:.4f} seconds")
    print(f"Period: {period1:.4f}, Significance: {significance1:.4f}")
    
    # Method 2: With preloaded objects
    start_time = time.time()
    period2, significance2 = calculate_period_with_preloaded_objects(oid, all_astro_objects, max_period=max_period)
    elapsed_time2 = time.time() - start_time
    print(f"\nMethod 2 (With preloading): {elapsed_time2:.4f} seconds")
    print(f"Period: {period2:.4f}, Significance: {significance2:.4f}")
    
    # Calculate speedup
    if elapsed_time1 > 0:
        speedup = elapsed_time1 / elapsed_time2
        print(f"\nSpeedup factor: {speedup:.2f}x")
    
    return {
        'Without preloading': elapsed_time1,
        'With preloading': elapsed_time2,
        'Speedup factor': speedup if elapsed_time1 > 0 else None
    }

# Example: Process multiple objects from a class using preloaded objects
def batch_process_with_preloaded_objects(class_name, num_objects=5, max_period=1000.0,):
    """
    Process multiple objects from a class using preloaded objects.
    
    Parameters:
    -----------
    class_name : str
        Class name to search for
    num_objects : int
        Number of objects to process
    max_period : float
        Maximum period to search for in days (default: 100.0)
    """
    print(f"Processing {num_objects} objects of class '{class_name}' with preloaded data (max_period={max_period})")
    
    import time
    start_time = time.time()
    
    results = process_objects_by_class_with_preloaded(class_name, num_objects, all_astro_objects, max_period=max_period)
    
    elapsed_time = time.time() - start_time
    print(f"\nProcessed {num_objects} objects in {elapsed_time:.2f} seconds")
    print(f"Average time per object: {elapsed_time / num_objects:.2f} seconds")
    
    return results

In [ ]:
# Example usage:
# 1. Select an OID to compare performance
if len(all_astro_objects) > 0:
    # Get a sample OID from the loaded objects
    sample_oid = list(all_astro_objects.keys())[0]
    
    # Compare performance with default max_period
    performance = compare_period_calculation_performance(sample_oid)
    
    # Compare performance with custom max_period
    # Uncomment to test different maximum periods:
    # performance_short = compare_period_calculation_performance(sample_oid, max_period=50.0)
    # performance_long = compare_period_calculation_performance(sample_oid, max_period=200.0)
    
    # Batch process multiple objects from a class with custom max_period
    # Uncomment to run:
    #results = batch_process_with_preloaded_objects('LPV', num_objects=3, max_period=1000.0)
    #display(results)

Comparing period calculation performance for OID ZTF18aaiexsc (max_period=1000.0)
Found OID ZTF18aaiexsc in batch file astro_objects_batch_006.pkl
